# OCT Masked Image Generator (OpenCV, no U-Net weights needed)
Run the cells from top to bottom in Google Colab.

## Step 1: Imports and settings
Change `INPUT_DIR` if your dataset folder has a different name.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Folder on your Drive that has  train/ and test/  (each with CNV, DME, DRUSEN, NORMAL)
INPUT_DIR  = "/content/drive/MyDrive/OCT2017_MODIFIED"
# New folder that will be created on your Drive for the masked images
OUTPUT_DIR = "/content/drive/MyDrive/OCT2017_MASKED"

CLASSES = ["CNV", "DME", "DRUSEN", "NORMAL"]
SPLITS  = ["train", "test"]
SIZE    = 224          # same size the paper uses

## Step 2: Mount Google Drive
A pop-up will ask you to allow access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Functions
These make the mask and apply it to the image. Just run this cell.

In [ ]:
def make_square(gray):
    """Put the image on a black square canvas (keeps aspect ratio), then resize."""
    h, w = gray.shape
    side = max(h, w)
    canvas = np.zeros((side, side), dtype=np.uint8)
    y0 = (side - h) // 2
    x0 = (side - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = gray
    return cv2.resize(canvas, (SIZE, SIZE))


def remove_white_background(gray):
    """Some scans have big white corners. Turn ONLY those big white areas black.
    (The bright retinal layer is also very bright, so we must not touch small bright bits.)"""
    white = (gray >= 250).astype(np.uint8) * 255
    contours, _ = cv2.findContours(white, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = gray.copy()
    for c in contours:
        if cv2.contourArea(c) > 2000:             # only large white regions
            cv2.drawContours(out, [c], -1, 0, thickness=cv2.FILLED)
    return out


def make_mask(gray):
    """Make a 0/255 mask of the retina from a 224x224 grayscale image."""
    # 1) Blur to kill the speckle / salt-and-pepper noise
    blur = cv2.medianBlur(gray, 5)
    blur = cv2.GaussianBlur(blur, (7, 7), 0)

    # 2) Otsu threshold = automatically picks "bright retina" vs "dark background"
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3) Opening removes small white dots, Closing fills small black holes
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # 4) Join nearby pieces so the retinal layers become one big shape
    mask = cv2.dilate(mask, np.ones((9, 9), np.uint8))

    # 5) Keep only big blobs (throw away tiny noise patches)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean = np.zeros_like(mask)
    for c in contours:
        if cv2.contourArea(c) > 1000:
            cv2.drawContours(clean, [c], -1, 255, thickness=cv2.FILLED)

    # 6) Slightly shrink back (undo the dilation) so edges stay tight
    clean = cv2.erode(clean, np.ones((5, 5), np.uint8))
    return clean


def process_image(path):
    """Read one image -> return (gray224, mask, masked_image). None if unreadable."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = remove_white_background(img)
    gray = make_square(img)
    mask = make_mask(gray)
    masked = cv2.bitwise_and(gray, gray, mask=mask)   # keep retina, black elsewhere
    return gray, mask, masked

## Step 4: Preview a few results
**Check this before the full run.** Each row shows original, mask, masked image.

In [ ]:
def preview(split="train"):
    fig, axes = plt.subplots(len(CLASSES), 3, figsize=(9, 3 * len(CLASSES)))
    for r, cls in enumerate(CLASSES):
        folder = os.path.join(INPUT_DIR, split, cls)
        fname = sorted(os.listdir(folder))[0]
        gray, mask, masked = process_image(os.path.join(folder, fname))
        for c, (im, title) in enumerate([(gray, cls + " original"), (mask, "mask"), (masked, "masked image")]):
            axes[r, c].imshow(im, cmap="gray")
            axes[r, c].set_title(title)
            axes[r, c].axis("off")
    plt.tight_layout()
    plt.show()

preview()

## Step 5: Run on the whole dataset
Saves masked images to `OCT2017_MASKED` on your Drive. Safe to re-run: finished files are skipped.

In [ ]:
for split in SPLITS:
    for cls in CLASSES:
        in_folder  = os.path.join(INPUT_DIR,  split, cls)
        out_folder = os.path.join(OUTPUT_DIR, split, cls)
        os.makedirs(out_folder, exist_ok=True)

        files = sorted(os.listdir(in_folder))
        done = 0
        for fname in files:
            out_path = os.path.join(out_folder, os.path.splitext(fname)[0] + ".png")
            if os.path.exists(out_path):          # already done earlier -> skip
                done += 1
                continue
            result = process_image(os.path.join(in_folder, fname))
            if result is None:
                print("could not read:", fname)
                continue
            cv2.imwrite(out_path, result[2])      # save the masked image
            done += 1
        print(f"{split}/{cls}: {done}/{len(files)} images saved")

print("Finished! Masked images are in:", OUTPUT_DIR)